# t-SNE & UMAP — Advanced Dimensionality Reduction for Visualization

**PCA** preserves global structure but misses non-linear relationships.  
**t-SNE** and **UMAP** are non-linear techniques that reveal clusters in high-dimensional data.

- **t-SNE**: t-Distributed Stochastic Neighbor Embedding
- **UMAP**: Uniform Manifold Approximation and Projection
- **PCA vs t-SNE vs UMAP** comparison
- **Hyperparameter effects** (perplexity, n_neighbors)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_digits, fetch_openml
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

try:
    import umap
    print(f'UMAP: {umap.__version__}')
except ImportError:
    print('Install: pip install umap-learn')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Digits dataset (8x8 images → 64 dimensions)
digits = load_digits()
X, y = digits.data, digits.target
print(f'Dataset: {X.shape} | Classes: {len(np.unique(y))}')

# Show sample digits
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(str(y[i]), fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Digits (64-dimensional data)', fontsize=13)
plt.tight_layout()
plt.show()

## 1. PCA — Linear Baseline

In [ ]:
start = time.time()
pca_result = PCA(n_components=2).fit_transform(X)
pca_time = time.time() - start

plt.figure(figsize=(8, 6))
scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1], c=y, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.title(f'PCA (2D) — Time: {pca_time:.3f}s')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. t-SNE

Algorithm:
1. Compute pairwise similarities in high-D (Gaussian kernel)
2. Initialize low-D points randomly
3. Compute pairwise similarities in low-D (t-distribution)
4. Minimize KL divergence between the two distributions

**Key parameter: `perplexity`** ≈ effective number of neighbors (5-50)

In [ ]:
start = time.time()
tsne_result = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000).fit_transform(X)
tsne_time = time.time() - start

plt.figure(figsize=(8, 6))
scatter = plt.scatter(tsne_result[:, 0], tsne_result[:, 1], c=y, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.title(f't-SNE (perplexity=30) — Time: {tsne_time:.2f}s')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Effect of perplexity
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, perp in zip(axes, [5, 15, 30, 50]):
    result = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000).fit_transform(X)
    ax.scatter(result[:, 0], result[:, 1], c=y, cmap='tab10', s=5, alpha=0.6)
    ax.set_title(f'Perplexity = {perp}')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('t-SNE — Effect of Perplexity', fontsize=14)
plt.tight_layout()
plt.show()

## 3. UMAP

**Faster** than t-SNE and preserves both **local AND global** structure.

Key parameters:
- `n_neighbors`: Local vs global structure (small = local, large = global)
- `min_dist`: How tightly packed points are (smaller = tighter clusters)

In [ ]:
start = time.time()
umap_result = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)
umap_time = time.time() - start

plt.figure(figsize=(8, 6))
scatter = plt.scatter(umap_result[:, 0], umap_result[:, 1], c=y, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.title(f'UMAP (n_neighbors=15) — Time: {umap_time:.2f}s')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Effect of n_neighbors and min_dist
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Vary n_neighbors
for ax, nn in zip(axes[0], [5, 15, 50]):
    result = umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=42).fit_transform(X)
    ax.scatter(result[:, 0], result[:, 1], c=y, cmap='tab10', s=5, alpha=0.6)
    ax.set_title(f'n_neighbors={nn}, min_dist=0.1')

# Vary min_dist
for ax, md in zip(axes[1], [0.0, 0.1, 0.5]):
    result = umap.UMAP(n_neighbors=15, min_dist=md, random_state=42).fit_transform(X)
    ax.scatter(result[:, 0], result[:, 1], c=y, cmap='tab10', s=5, alpha=0.6)
    ax.set_title(f'n_neighbors=15, min_dist={md}')

plt.suptitle('UMAP — Hyperparameter Effects', fontsize=14)
plt.tight_layout()
plt.show()

## 4. PCA vs t-SNE vs UMAP — Side by Side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
results = [('PCA', pca_result, pca_time), ('t-SNE', tsne_result, tsne_time), ('UMAP', umap_result, umap_time)]

for ax, (name, result, t) in zip(axes, results):
    scatter = ax.scatter(result[:, 0], result[:, 1], c=y, cmap='tab10', s=8, alpha=0.7)
    ax.set_title(f'{name} ({t:.2f}s)', fontsize=13)
    ax.set_xticks([]); ax.set_yticks([])

plt.colorbar(scatter, ax=axes, label='Digit', shrink=0.8)
plt.suptitle('Dimensionality Reduction — PCA vs t-SNE vs UMAP', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Comparison Summary

| Feature | PCA | t-SNE | UMAP |
|---------|-----|-------|------|
| **Type** | Linear | Non-linear | Non-linear |
| **Speed** | Fastest | Slowest | Fast |
| **Global Structure** | Yes | No | **Yes** |
| **Local Structure** | No | **Yes** | **Yes** |
| **Scalability** | Excellent | Poor (>10K rows) | Good |
| **Deterministic** | Yes | No | No |
| **Use for ML** | Yes (preprocessing) | Visualization only | Both |

### When to Use:
- **PCA**: Preprocessing, variance analysis, when interpretability matters
- **t-SNE**: Publication-quality cluster visualizations (<10K samples)
- **UMAP**: Large datasets, when speed matters, preserving global structure

### Caution
- **t-SNE cluster sizes don't mean anything** — distances between clusters are not meaningful
- Always try **multiple perplexity/n_neighbors** values
- Use **PCA first** to reduce to 50 dims, then apply t-SNE/UMAP